# Mutatierapport Etten-Leur: 2022 → 2025 — **versie 2**

Zelfde rapport als `mutaties_etten_leur.ipynb`, met twee methodische verbeteringen
(dat notebook blijft ongewijzigd zodat je eerdere run vergelijkbaar blijft):

- **Contourmasker**: de verschilscore telt alleen pixels bínnen de pandcontour —
  geen erf of akker meer in de meting (v1 gebruikte de omsluitende box).
- **Drempel per omgevingstype**: panden worden ingedeeld in *stedelijk* of
  *buitengebied* (panddichtheid per tegel) en de detectiedrempel wordt per stratum
  bepaald, zodat natuurlijk sterker veranderend buitengebied de stedelijke detecties
  niet verdringt. Sectie 3 toont beide verdelingen als bewijs.

De scores van v2 zijn hierdoor **niet één-op-één vergelijkbaar** met v1 — draai de
rekencel opnieuw (5–15 min) en lees de trechter en drempelkeuze-curves opnieuw af.

**Methode**: BAG-mutaties (bouwjaar ≥ 2022, statussen) + visuele verschilscore per pand
(verschuivings-tolerant tegen omvalling), gekruist tot *verklaard door BAG* en
*onverklaard* (werkvoorraad). Uitleg: [docs/verschilscore-techniek.md](../docs/verschilscore-techniek.md).

Vereiste downloads (herstartbaar, samen ± 1 GB; heb je ze al, dan wordt niets opnieuw gedownload):
```
python scripts/02_download_tiles.py --bbox 101500,395000,107000,400000 --laag 2022_orthoHR --map-naam el_2022
python scripts/02_download_tiles.py --bbox 101500,395000,107000,400000 --laag 2025_orthoHR --map-naam el_2025
```

In [ ]:
import io, json, sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from shapely.geometry import shape

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'scripts'))
from common import laad_config
from pdok import fetch_bag_panden, make_session, wms_get_map

cfg = laad_config(REPO / 'config.yaml')
DATA = REPO / cfg['paden']['data']
RES = cfg['luchtfoto']['resolutie']
PX = cfg['luchtfoto']['tegelgrootte']
STAP = PX * RES

GEBIED = [101500, 395000, 107000, 400000]     # heel Etten-Leur
OUD_LAAG, NIEUW_LAAG = '2022_orthoHR', '2025_orthoHR'
OUD_MAP, NIEUW_MAP = DATA / 'el_2022', DATA / 'el_2025'
BAG_ALLE = DATA / 'bag' / 'el_panden_alle.geojson'
DREMPEL_PCT = 99          # percentiel van de verschilscore dat als 'detectie' telt
MARKEER = '#FFD400'
BLAUW, GRIJS, AMBER = '#2563EB', '#94A3B8', '#F59E0B'
sessie = make_session()

for m in (OUD_MAP, NIEUW_MAP):
    assert (m / 'tiles.json').exists(), f'{m} ontbreekt — zie de downloadregels hierboven'
idx_oud = json.loads((OUD_MAP / 'tiles.json').read_text())
idx_nieuw = json.loads((NIEUW_MAP / 'tiles.json').read_text())

if not BAG_ALLE.exists():
    feats = fetch_bag_panden(sessie, cfg['bag']['wfs_url'], tuple(GEBIED), alleen_in_gebruik=False)
    BAG_ALLE.write_text(json.dumps({'type': 'FeatureCollection', 'bbox_rd': GEBIED, 'features': feats}))
panden = json.loads(BAG_ALLE.read_text())['features']
print(f'{len(panden)} BAG-panden, {len(set(idx_oud) & set(idx_nieuw))} tegelparen geladen')

## 1. Wat de BAG registreert (2022–2025)

In [ ]:
nieuwbouw = sorted((f for f in panden if 2022 <= (f['properties'].get('bouwjaar') or 0) <= 2025),
                   key=lambda f: f['properties']['bouwjaar'])
in_aanbouw = [f for f in panden if (f['properties'].get('bouwjaar') or 0) >= 2026
              or f['properties'].get('status') == 'Bouw gestart']
sloop = [f for f in panden if 'loop' in (f['properties'].get('status') or '')]
verbouwing = [f for f in panden if f['properties'].get('status') == 'Verbouwing pand']
vergund = [f for f in panden if f['properties'].get('status') == 'Bouwvergunning verleend']

per_jaar = Counter(f['properties']['bouwjaar'] for f in nieuwbouw)
jaren = sorted(per_jaar)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar([str(j) for j in jaren], [per_jaar[j] for j in jaren], color=BLAUW, width=0.55)
for i, j in enumerate(jaren):
    ax.text(i, per_jaar[j] + max(per_jaar.values()) * 0.02, str(per_jaar[j]),
            ha='center', color='#555555', fontsize=9)
ax.set_title('Nieuwbouw in Etten-Leur per bouwjaar (BAG)', loc='left', fontsize=11)
ax.set_ylabel('panden')
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=0)
plt.tight_layout(); plt.show()

print(f'Nieuw gebouwd 2022-2025:  {len(nieuwbouw)} panden')
print(f'In aanbouw (2025-foto):   {len(in_aanbouw)} panden')
print(f'Sloopvergunning verleend: {len(sloop)} panden')
print(f'Verbouwing pand:          {len(verbouwing)} panden')
print(f'Bouwvergunning verleend:  {len(vergund)} panden')

## 2. Visuele verschilscore per pand (verschuivings-tolerant)

Per tegelpaar: licht blurren, per tegel normaliseren (z-score, dempt lichtverschil), en per
pixel de kleinste afwijking nemen over 9 verschuivingen van ±8 px — zo telt een dak dat door
omvalling een halve meter "verschoven" lijkt niet als verandering. De score per pand is het
gemiddelde binnen de pandcontour-box. Reken op enkele minuten voor de hele stad.

Volledige uitleg van de techniek (en waarom we níet eerst aftrekken en dan pas rekenen):
[docs/verschilscore-techniek.md](../docs/verschilscore-techniek.md).

In [ ]:
GRID_X, GRID_Y = GEBIED[0], GEBIED[1]
def tegel_id_voor(x, y):
    return f't_{int((x - GRID_X) // STAP):04d}_{int((y - GRID_Y) // STAP):04d}'

def laad_genorm(pad):
    beeld = Image.open(pad).convert('L').filter(ImageFilter.GaussianBlur(1.5))
    a = np.asarray(beeld, dtype=np.float32)
    return (a - a.mean()) / (a.std() + 1e-6)

per_tegel = defaultdict(list)
for f in panden:
    geom = shape(f['geometry'])
    if geom.area < 25:
        continue
    per_tegel[tegel_id_voor(geom.centroid.x, geom.centroid.y)].append((f, geom))

# Omgevingstype per tegel: weinig panden = buitengebied (akkers/erven veranderen
# van nature sterker; daarom krijgt elk stratum straks zijn eigen drempel).
STEDELIJK_VANAF = 6   # panden per tegel

paren = [tid for tid in per_tegel if tid in idx_oud and tid in idx_nieuw]
VERSCHUIVINGEN = [(dx, dy) for dx in (-8, 0, 8) for dy in (-8, 0, 8)]
scores = []
stratum_per_id = {}
from tqdm.auto import tqdm
for tid in tqdm(paren, desc='tegelparen scoren'):
    a = laad_genorm(OUD_MAP / idx_oud[tid]['image'])
    b = laad_genorm(NIEUW_MAP / idx_nieuw[tid]['image'])
    minverschil = np.full_like(a, np.inf)
    for dx, dy in VERSCHUIVINGEN:
        d = np.abs(np.roll(b, (dy, dx), axis=(0, 1)) - a)
        np.minimum(minverschil, d, out=minverschil)
    bbox = idx_oud[tid]['bbox']
    stratum = 'stedelijk' if idx_oud[tid].get('n_panden', 0) >= STEDELIJK_VANAF else 'buitengebied'
    for f, geom in per_tegel[tid]:
        gx0, gy0, gx1, gy1 = geom.bounds
        x0 = max(0, int((gx0 - bbox[0]) / RES)); x1 = min(PX, int((gx1 - bbox[0]) / RES))
        y0 = max(0, int((bbox[3] - gy1) / RES)); y1 = min(PX, int((bbox[3] - gy0) / RES))
        if x1 - x0 < 12 or y1 - y0 < 12:
            continue
        # Alleen pixels bínnen de pandcontour tellen mee — geen erf of akker in de box.
        masker = Image.new('1', (x1 - x0, y1 - y0), 0)
        tekenaar = ImageDraw.Draw(masker)
        for poly in (geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]):
            punten = [((px - bbox[0]) / RES - x0, (bbox[3] - py) / RES - y0)
                      for px, py in poly.exterior.coords]
            tekenaar.polygon(punten, fill=1)
        binnen = np.asarray(masker, dtype=bool)
        venster = minverschil[y0:y1, x0:x1]
        s = float(venster[binnen].mean()) if binnen.sum() >= 100 else float(venster.mean())
        scores.append((s, f, geom))
        stratum_per_id[f['properties']['identificatie']] = stratum

n_sted = sum(1 for v in stratum_per_id.values() if v == 'stedelijk')
print(f'{len(scores)} panden gescoord over {len(paren)} tegelparen '
      f'({n_sted} stedelijk, {len(scores) - n_sted} buitengebied)')

## 3. Resultaat: gedetecteerd, verklaard door BAG, onverklaard

In [ ]:
waarden = np.array([s for s, _, _ in scores])
strata = np.array([stratum_per_id[f['properties']['identificatie']] for _, f, _ in scores])
score_per_id = {f['properties']['identificatie']: s for s, f, _ in scores}

# Drempel PER omgevingstype: akkers/erven veranderen van nature sterker, dus één
# stadsbrede drempel zou het buitengebied oververtegenwoordigen in de werkvoorraad.
def drempels_bij(pct):
    return {naam: float(np.percentile(waarden[strata == naam], pct))
            for naam in ('stedelijk', 'buitengebied') if (strata == naam).any()}

drempel_per = drempels_bij(DREMPEL_PCT)

def is_detectie(s, pid, drempels=None):
    return s >= (drempels or drempel_per)[stratum_per_id[pid]]

bag_verklaard_ids = set()
for groep in (nieuwbouw, in_aanbouw, sloop, verbouwing, vergund):
    bag_verklaard_ids |= {f['properties']['identificatie'] for f in groep}

detecties = [(s, f, g) for s, f, g in scores
             if is_detectie(s, f['properties']['identificatie'])]
verklaard = [d for d in detecties if d[1]['properties']['identificatie'] in bag_verklaard_ids]
onverklaard = [d for d in detecties if d[1]['properties']['identificatie'] not in bag_verklaard_ids]
onverklaard.sort(key=lambda d: -d[0])

bewaar_map = DATA / 'mutaties_preview'
bewaar_map.mkdir(exist_ok=True)

# Hypothese-check: verschillen de verdelingen tussen stad en buitengebied?
fig, ax = plt.subplots(figsize=(7.5, 3.2))
for naam, kleur in (('stedelijk', BLAUW), ('buitengebied', AMBER)):
    deel = waarden[strata == naam]
    if not len(deel):
        continue
    ax.hist(deel, bins=60, color=kleur, alpha=0.65, density=True,
            label=f'{naam} (mediaan {np.median(deel):.2f}, drempel {drempel_per[naam]:.2f})')
    ax.axvline(drempel_per[naam], color=kleur, linestyle='--', linewidth=1.2)
ax.legend(frameon=False, fontsize=9)
ax.set_title('Verschilscores per omgevingstype, elk met eigen drempel', loc='left', fontsize=11)
ax.set_xlabel('verschilscore'); ax.set_ylabel('dichtheid')
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=0)
plt.tight_layout(); plt.show()

# Trechter: van scan naar werkvoorraad
stappen = [('Panden geanalyseerd', len(scores), GRIJS),
           (f'Visueel gedetecteerd (≥ P{DREMPEL_PCT} van eigen stratum)', len(detecties), BLAUW),
           ('Verklaard door BAG-registratie', len(verklaard), BLAUW),
           ('Onverklaard → werkvoorraad taxateur', len(onverklaard), AMBER)]
fig, ax = plt.subplots(figsize=(7.5, 2.8))
posities = range(len(stappen))
ax.barh(posities, [n for _, n, _ in stappen], height=0.55, color=[k for _, _, k in stappen])
ax.set_yticks(posities, [naam for naam, _, _ in stappen])
ax.invert_yaxis()
ax.set_xscale('log')
for i, (_, n, _) in enumerate(stappen):
    ax.text(n * 1.12, i, str(n), va='center', color='#555555', fontsize=9)
ax.set_title('Van scan naar werkvoorraad (logaritmische as)', loc='left', fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=0)
plt.tight_layout()
fig.savefig(bewaar_map / 'trechter.jpg', dpi=110, bbox_inches='tight')
plt.show()

## 3b. Omgekeerde controle: hoeveel bekende BAG-mutaties vangt de visuele score?

De trechter hierboven kijkt één kant op (visuele detecties → verklaard door BAG). Deze
sectie kijkt andersom en gebruikt de BAG-mutaties als **gratis grondwaarheid** om de
*recall* van de visuele score te meten. Een gemiste BAG-mutatie is geen probleem voor de
werkvoorraad (die mutatie kénde je al), maar wél de beste maat om de drempel te kalibreren.

**Let op de foto-timing bij het interpreteren.** De foto's zijn winteropnames van *begin*
2022 en *begin* 2025:

- **Bouwjaar 2023–2024** zit gegarandeerd tussen de foto's → hoort boven de drempel;
  dít is de eerlijke meetlat (de drempelkeuze-cel hieronder gebruikt alleen deze groep).
- **Bouwjaar 2022** stond deels al op de eerste foto; **2025/2026** staat vaak nog niet
  op de tweede — lage scores zijn daar terecht, geen gemiste detecties.
- **Bouwvergunning verleend** hoort onder de drempel (nog niets gebouwd); **verbouwing**
  kan inwendig zijn; **sloopvergunning** betekent nog niet gesloopt.
- Overige verwachte missers: panden < 25 m² (niet gescoord) en panden buiten de tegeldekking.

In [ ]:
categorieen = [('Nieuwbouw 2022-2025', nieuwbouw),
               ('In aanbouw', in_aanbouw),
               ('Sloopvergunning', sloop),
               ('Verbouwing', verbouwing),
               ('Bouwvergunning (nog niet gebouwd)', vergund)]

print(f'{"categorie":<34}{"totaal":>7}{"gescoord":>10}{"gedetecteerd":>13}   recall')
staaf_data = []
for naam, groep in categorieen:
    ids = [f['properties']['identificatie'] for f in groep]
    gescoord = [i for i in ids if i in score_per_id]
    boven = [i for i in gescoord if is_detectie(score_per_id[i], i)]
    recall = len(boven) / len(gescoord) if gescoord else 0
    print(f'{naam:<34}{len(ids):>7}{len(gescoord):>10}{len(boven):>13}   {recall:.0%}')
    staaf_data.append((naam, len(boven), len(gescoord) - len(boven), len(ids) - len(gescoord)))

fig, ax = plt.subplots(figsize=(8.5, 3.2))
posities = range(len(staaf_data))
links = np.zeros(len(staaf_data))
for kleur, label, kolom in ((BLAUW, 'boven drempel (gedetecteerd)', 1),
                            (GRIJS, 'onder drempel (visueel gemist)', 2),
                            ('#E2E8F0', 'niet gescoord (buiten dekking / te klein)', 3)):
    deel = [r[kolom] for r in staaf_data]
    ax.barh(posities, deel, left=links, height=0.55, color=kleur, label=label,
            edgecolor='white', linewidth=1)
    links += deel
ax.set_yticks(posities, [r[0] for r in staaf_data])
ax.invert_yaxis()
ax.legend(frameon=False, fontsize=8, loc='lower right')
ax.set_title('Vangt de visuele score de bekende BAG-mutaties?', loc='left', fontsize=11)
ax.set_xlabel('panden')
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=0)
plt.tight_layout()
fig.savefig(bewaar_map / 'recall_bag.jpg', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# Drempelkeuze op data: recall op gegarandeerd zichtbare mutaties vs. werkvoorraad.
# Meetlat: alleen bouwjaar 2023-2024 (winteropnames van BEGIN 2022 en BEGIN 2025:
# bouwjaar 2022 stond deels al op de eerste foto, 2025/2026 nog niet op de tweede).
# De drempel wordt per omgevingstype toegepast (stedelijk/buitengebied).
# Kies het percentiel, zet DREMPEL_PCT (setup-cel) en draai vanaf sectie 3 opnieuw —
# de rekencel hoeft NIET opnieuw.
zichtbaar_ids = [f['properties']['identificatie'] for f in nieuwbouw
                 if 2023 <= f['properties']['bouwjaar'] <= 2024]
zichtbaar_ids = [i for i in zichtbaar_ids if i in score_per_id]
print(f'Meetlat: {len(zichtbaar_ids)} gescoorde panden met bouwjaar 2023-2024')

kandidaat_pct = np.arange(85, 99.6, 0.5)
recalls, werkvoorraad = [], []
for p in kandidaat_pct:
    dr = drempels_bij(p)
    recalls.append(100 * float(np.mean([is_detectie(score_per_id[i], i, dr)
                                        for i in zichtbaar_ids])))
    n_det = n_verkl = 0
    for s, f, _ in scores:
        pid = f['properties']['identificatie']
        if is_detectie(s, pid, dr):
            n_det += 1
            if pid in bag_verklaard_ids:
                n_verkl += 1
    werkvoorraad.append(n_det - n_verkl)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 3.2))
ax1.plot(kandidaat_pct, recalls, color=BLAUW, linewidth=2)
ax1.set_title('Gevangen deel van nieuwbouw 2023-2024', loc='left', fontsize=10)
ax1.set_xlabel('drempel (percentiel per stratum)'); ax1.set_ylabel('% gevangen')
ax2.plot(kandidaat_pct, werkvoorraad, color=AMBER, linewidth=2)
ax2.set_title('Omvang werkvoorraad (onverklaarde detecties)', loc='left', fontsize=10)
ax2.set_xlabel('drempel (percentiel per stratum)'); ax2.set_ylabel('panden')
for ax in (ax1, ax2):
    ax.axvline(DREMPEL_PCT, color='#555555', linestyle='--', linewidth=1)
    ax.text(DREMPEL_PCT, ax.get_ylim()[1] * 0.05, f' huidig: P{DREMPEL_PCT}',
            color='#555555', fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(length=0)
plt.tight_layout()
fig.savefig(bewaar_map / 'drempelkeuze.jpg', dpi=110, bbox_inches='tight')
plt.show()

for p in (90, 95, 99):
    dr = drempels_bij(p)
    r = float(np.mean([is_detectie(score_per_id[i], i, dr) for i in zichtbaar_ids]))
    print(f'P{p}: drempels {dr} -> {r:.0%} van nieuwbouw 2023-2024 gevangen')

## 4. Voor/na-galerijen (gele box = het pand in kwestie)

In [ ]:
def beeldpaar_rond(geom):
    cx, cy = geom.centroid.x, geom.centroid.y
    tid = tegel_id_voor(cx, cy)
    if tid in idx_oud and tid in idx_nieuw:
        return (Image.open(OUD_MAP / idx_oud[tid]['image']).convert('RGB'),
                Image.open(NIEUW_MAP / idx_nieuw[tid]['image']).convert('RGB'),
                idx_oud[tid]['bbox'])
    halve = STAP / 2
    bbox = (cx - halve, cy - halve, cx + halve, cy + halve)
    oud, nieuw = (Image.open(io.BytesIO(wms_get_map(sessie, cfg['luchtfoto']['wms_url'],
                                                    laag, bbox, PX, PX))).convert('RGB')
                  for laag in (OUD_LAAG, NIEUW_LAAG))
    return oud, nieuw, list(bbox)

def markeer_en_crop(beeld, geom, bbox, marge_m=14):
    gx0, gy0, gx1, gy1 = geom.bounds
    x0, x1 = (gx0 - bbox[0]) / RES, (gx1 - bbox[0]) / RES
    y0, y1 = (bbox[3] - gy1) / RES, (bbox[3] - gy0) / RES
    kopie = beeld.copy()
    ImageDraw.Draw(kopie).rectangle([x0, y0, x1, y1], outline=MARKEER, width=4)
    m = marge_m / RES
    return kopie.crop((max(0, x0 - m), max(0, y0 - m), min(PX, x1 + m), min(PX, y1 + m)))

def toon_voor_na(rijen, titel, bewaar=None):
    if not rijen:
        print('(geen voorbeelden)'); return
    fig, assen = plt.subplots(len(rijen), 2, figsize=(9, 4.4 * len(rijen)), squeeze=False)
    for (geom, onderschrift), (as_o, as_n) in zip(rijen, assen):
        oud, nieuw, bbox = beeldpaar_rond(geom)
        as_o.imshow(markeer_en_crop(oud, geom, bbox))
        as_n.imshow(markeer_en_crop(nieuw, geom, bbox))
        as_o.set_title(f'2022 — {onderschrift}', fontsize=9, loc='left')
        as_n.set_title('2025', fontsize=9, loc='left')
        as_o.axis('off'); as_n.axis('off')
    fig.suptitle(titel, fontsize=12)
    plt.tight_layout()
    if bewaar:
        fig.savefig(bewaar, dpi=110, bbox_inches='tight')
    plt.show()

print('Helpers klaar.')

In [ ]:
# Onverklaarde detecties: hoogste scores eerst. Meer zien? Verhoog N_VOORBEELDEN.
N_VOORBEELDEN = 12
for start in range(0, min(N_VOORBEELDEN, len(onverklaard)), 4):
    groep = onverklaard[start:start + 4]
    rijen = [(g, f'verschilscore {s:.2f}, bouwjaar {f["properties"].get("bouwjaar")}')
             for s, f, g in groep]
    toon_voor_na(rijen, f'Onverklaarde veranderingen — {start + 1} t/m {start + len(groep)}',
                 bewaar=(bewaar_map / 'onverklaard.jpg') if start == 0 else None)

In [ ]:
grootste = sorted(nieuwbouw, key=lambda f: -(f['properties'].get('oppervlakte_max') or 0))[:3]
rijen = [(shape(f['geometry']),
          f'bouwjaar {f["properties"]["bouwjaar"]}, {f["properties"].get("oppervlakte_max")} m²')
         for f in grootste]
toon_voor_na(rijen, 'Nieuwbouw — grootste drie (ter validatie van de methode)',
             bewaar=bewaar_map / 'nieuwbouw.jpg')

In [ ]:
# Ter controle: BAG-nieuwbouw die visueel ONDER de drempel bleef — waarom gemist?
# Laagste scores eerst (de duidelijkst gemiste gevallen). Meer zien? Verhoog N_GEMIST.
N_GEMIST = 6
nb_scores = [(score_per_id.get(f['properties']['identificatie']), f) for f in nieuwbouw]
gemist = sorted(((s, f) for s, f in nb_scores
                 if s is not None and not is_detectie(s, f['properties']['identificatie'])),
                key=lambda x: x[0])
ongescoord = sum(1 for s, _ in nb_scores if s is None)
for start in range(0, min(N_GEMIST, len(gemist)), 3):
    groep = gemist[start:start + 3]
    rijen = [(shape(f['geometry']),
              f'bouwjaar {f["properties"]["bouwjaar"]}, score {s:.2f} '
              f'(drempel {drempel_per[stratum_per_id[f["properties"]["identificatie"]]]:.2f}, '
              f'{stratum_per_id[f["properties"]["identificatie"]]})')
             for s, f in groep]
    toon_voor_na(rijen, f'Gemiste BAG-nieuwbouw — {start + 1} t/m {start + len(groep)}')
print(f'{len(gemist)} van de {len(nieuwbouw)} nieuwbouwpanden bleef onder de drempel; '
      f'{ongescoord} niet gescoord (buiten dekking of < 25 m²).')

## 5. Samenvatting

In [ ]:
dekking = len(paren) / max(1, len(per_tegel))
opp = sum(f['properties'].get('oppervlakte_max') or 0 for f in nieuwbouw)
drempels_txt = ', '.join(f'{naam} {d:.2f}' for naam, d in drempel_per.items())
print(f'Etten-Leur (5,5 x 5 km), luchtfoto 2022 -> 2025:')
print(f'- {len(scores)} panden visueel geanalyseerd ({dekking:.0%} tegeldekking)')
print(f'- BAG: {len(nieuwbouw)} nieuw gebouwd (± {opp} m²), {len(in_aanbouw)} in aanbouw,')
print(f'       {len(sloop)} sloopvergunningen, {len(verbouwing)} verbouwingen, {len(vergund)} bouwvergunningen')
print(f'- {len(detecties)} visuele detecties boven P{DREMPEL_PCT} van het eigen stratum '
      f'(drempels: {drempels_txt})')
print(f'  waarvan {len(verklaard)} verklaard door de BAG en {len(onverklaard)} ONVERKLAARD')
print(f'- De onverklaarde lijst is de werkvoorraad: vooral zonnepanelen, dakrenovaties en')
print(f'  niet-geregistreerde bouwwerken. Volgende stap: bevestigen/afwijzen in Label Studio.')

---
**Kanttekening bij de methode**: de verschuivings-tolerante score dempt omvalling maar
elimineert hem niet — hoge gebouwen aan de rand van een vluchtstrook kunnen boven de
drempel uitkomen zonder echte verandering. De structurele oplossing is het getrainde
verandermodel ([docs/mutatiedetectie.md](../docs/mutatiedetectie.md)); de bevestigde en
afgewezen detecties uit dít rapport zijn daar de trainingsdata voor.

*Bevat gegevens van PDOK: Luchtfoto Beeldmateriaal Nederland (CC-BY 4.0) en de BAG.*